# z instead of y on PPBench

GAR's restart with its random directions written into the fillable cells of `z` instead of `y`, for up to 64 retries
on the validation and golden splits of PPBench, with three seeds. It is compared with GAR, whose passes are read from
the cache of `PPBench_GAR` (and run here if missing), and with PTRM under the same harness, whose rollouts are read
from the cache of `PPBench_continue_and_PTRM`.

Set `CKPT_TRM`, `DATA_PATH` and `CACHE_DIR` below. Each pass is saved in `CACHE_DIR` and reloaded when the notebook
runs again.

In [ ]:
!pip -q install einops pydantic argdantic
!test -d TRM || git clone -q https://github.com/SamsungSAILMontreal/TinyRecursiveModels TRM
!git -C TRM checkout -q c011037

In [ ]:
import os, sys, ast, json, dataclasses

import numpy as np
import pandas as pd
import torch
import yaml

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

CKPT_TRM = "/content/drive/MyDrive/GAR/ppbench/checkpoints/step_254961"  # with all_config.yaml next to it
DATA_PATH = "/content/drive/MyDrive/GAR/ppbench/data/ppbench-6type"  # built by PPBench_data_and_training
CACHE_DIR = "/content/drive/MyDrive/GAR/ppbench/cache"
GLOBAL_BATCH_SIZE = 768
DEVICE = "cuda"
REPORT_TYPES = ["sudoku", "lightup", "nurikabe", "heyawake", "tapa"]  # shakashaka is left out of the aggregates

sys.path.insert(0, os.path.abspath("TRM"))
from models.losses import ACTLossHead, IGNORE_LABEL_ID
from models.recursive_reasoning.trm import TinyRecursiveReasoningModel_ACTV1
from puzzle_dataset import PuzzleDataset, PuzzleDatasetConfig

os.makedirs(CACHE_DIR, exist_ok=True)
pd.options.display.float_format = "{:.2f}".format

## Checkpoint, test sets and fillable cells

The test split has one set per puzzle type and split, `val_<type>` and `gold_<type>`, each loaded as one batch. The
model is compiled, as for the paper's numbers. PPBench has no blank token: the fillable cells of a puzzle are the
cells whose token does not belong to a family of clues fixed by the input (`STATIC_FAMILIES`). The check below
confirms, on the labels, that this mask contains every cell whose solution differs from the input and no unlabelled
cell.

In [ ]:
test_set = PuzzleDataset(PuzzleDatasetConfig(seed=0, dataset_paths=[DATA_PATH], global_batch_size=GLOBAL_BATCH_SIZE,
                                             test_set_mode=True, epochs_per_iter=1, rank=0, num_replicas=1),
                         split="test")
BATCHES = [(name, {k: v.to(DEVICE) for k, v in batch.items()}) for name, batch, _ in test_set]

with open(os.path.join(os.path.dirname(CKPT_TRM), "all_config.yaml")) as f:
    arch = yaml.safe_load(f)["arch"]
loss_type = arch.pop("loss")["loss_type"]
del arch["name"]
state = {k.replace("_orig_mod.", "", 1): v for k, v in torch.load(CKPT_TRM, map_location=DEVICE).items()}
config = dict(arch, batch_size=GLOBAL_BATCH_SIZE, vocab_size=test_set.metadata.vocab_size,
              seq_len=test_set.metadata.seq_len,
              num_puzzle_identifiers=state["model.inner.puzzle_emb.weights"].shape[0])
with torch.device(DEVICE):
    model = ACTLossHead(TinyRecursiveReasoningModel_ACTV1(config), loss_type=loss_type)
model.load_state_dict(state, strict=False, assign=True)
model = torch.compile(model.eval())  # halting is off: every block runs exactly T supervision steps
inner = model.model.inner
T = model.model.config.halt_max_steps  # supervision steps per block (16)
P = inner.puzzle_emb_len  # prefix positions before the 100 cells (16)

STATIC_FAMILIES = {"sudoku": {"g"}, "lightup": {"wall"}, "shakashaka": {"wall"}, "nurikabe": {"n"}, "tapa": {"c"},
                   "heyawake": set()}
with open(os.path.join(DATA_PATH, "vocab.json")) as f:
    VOCAB = {ast.literal_eval(k): v for k, v in json.load(f).items()}  # (type, clue, mark[, walls]) -> token
FILL_IDS = torch.tensor(sorted(v for (pid, clue, *_), v in VOCAB.items()
                               if (clue[0] if isinstance(clue, tuple) and clue else str(clue))
                               not in STATIC_FAMILIES.get(pid, set())), device=DEVICE, dtype=torch.int32)


def fillable(batch):
    """The fillable cells, read from the input tokens only."""
    return torch.isin(batch["inputs"], FILL_IDS.to(batch["inputs"].dtype))


for _, batch in BATCHES:
    labelled = batch["labels"] != IGNORE_LABEL_ID
    assert not (fillable(batch) & ~labelled).any()
    assert not ((batch["labels"] != batch["inputs"]) & labelled & ~fillable(batch)).any()


def split_sets(split):
    """The sets of a split that enter its aggregate, in the order used to concatenate puzzles."""
    return sorted(name for name, _ in BATCHES if name.startswith(split + "_") and name.split("_", 1)[1] in REPORT_TYPES)

## The harness

The standard pass runs one block of `T` supervision steps. The gate re-opens a puzzle whose halting logit `q` is
negative, and a retry runs one more block on it. The search stops at the first block with `q >= 0`, or after
`ROUNDS` retries, and returns the grid of highest `q` among the blocks run (the fallback). A retry returns `y`
(`z_H` in TRM's code) and `z` (`z_L`) to their initial states, then writes isotropic random directions into the
fillable cells of `y` (GAR, `restart`) or of `z` (`restart_mirror`). Both arms draw the same directions.

In [ ]:
Q_THR = 0.0  # the gate re-opens a puzzle when q < Q_THR, and a block with q >= Q_THR stops the search
SEEDS = [0, 1, 2]


def clone(x):
    """Deep copy of a TRM carry."""
    if torch.is_tensor(x):
        return x.clone()
    if dataclasses.is_dataclass(x):
        return type(x)(**{f.name: clone(getattr(x, f.name)) for f in dataclasses.fields(x)})
    if isinstance(x, dict):
        return {k: clone(v) for k, v in x.items()}
    return x


def correct(pred, batch):
    """Exact match on the labelled cells, as counted by TRM's evaluator, and the rows that have labels."""
    mask = batch["labels"] != IGNORE_LABEL_ID
    return (mask & (pred == batch["labels"])).sum(-1) == mask.sum(-1), mask.any(-1)


@torch.no_grad()
def run_block(batch, carry=None):
    """One block of T supervision steps; carry=None starts from the initial states (the standard pass)."""
    if carry is None:
        with torch.device(DEVICE):
            carry = model.initial_carry(batch)
    for _ in range(T):
        carry, _, _, out, _ = model(carry=carry, batch=batch, return_keys={"preds", "q_halt_logits"})
    return carry, out["preds"], out["q_halt_logits"].float()



ROUNDS = 64


@torch.no_grad()
def retry(carry, fill, active, arm, gen):
    """Prepares a retry of the rows `active`: returns y and z to their initial states on every position, then writes
    isotropic directions into the fillable cells of y (restart) or z (restart_mirror), at the norm of its initial
    state."""
    zH, zL = carry.inner_carry.z_H, carry.inner_carry.z_L
    e = torch.randn(zH[:, P:].shape, generator=gen, device=zH.device, dtype=torch.float32)
    e = e / e.norm(dim=-1, keepdim=True).clamp_min(1e-9)
    rows = active.view(-1, 1, 1)
    zH.copy_(torch.where(rows, inner.H_init.to(zH.dtype), zH))
    zL.copy_(torch.where(rows, inner.L_init.to(zL.dtype), zL))
    state, init = (zH, inner.H_init) if arm == "restart" else (zL, inner.L_init)
    cells = (fill & active.view(-1, 1)).unsqueeze(-1)
    state[:, P:] = torch.where(cells, (e * init.float().norm()).to(state.dtype), state[:, P:])
    carry.halted.fill_(False)  # so that the next step continues from this state instead of resetting it
    carry.steps.zero_()


@torch.no_grad()
def run_pass(arm, seed):
    """The harness on every test set with retries of `arm`. Per set and per puzzle: the standard pass (ok0, q0, pred0),
    ok[:, r] (the returned grid is correct after r retries) and act[:, r] (the puzzle is open before retry r + 1)."""
    res = {}
    for b, (name, batch) in enumerate(BATCHES):
        fill = fillable(batch)
        gen = torch.Generator(device=DEVICE).manual_seed(seed * 100003 + b)
        carry, pred0, q0 = run_block(batch)
        ok0, valid = correct(pred0, batch)
        ok0 &= valid
        best_p, best_q = pred0.clone(), q0.clone()
        active = valid & (q0 < Q_THR)
        ok, act = [ok0], []
        for _ in range(ROUNDS):
            act.append(active.clone())
            if active.any():
                retry(carry, fill, active, arm, gen)
                carry, p, q = run_block(batch, carry)
                better = active & (q > best_q)
                best_p = torch.where(better.view(-1, 1), p, best_p)
                best_q = torch.where(better, q, best_q)
                active &= q < Q_THR
            ok.append(correct(best_p, batch)[0] & valid)
        rows = valid.nonzero(as_tuple=True)[0]  # drops the padding rows of the batch
        res[name] = {k: v[rows].cpu().numpy() for k, v in (("ok0", ok0), ("q0", q0), ("pred0", pred0),
                                                            ("ok", torch.stack(ok, 1)), ("act", torch.stack(act, 1)))}
    return res

## The cache and a self-test

In [ ]:
# The standard pass on every test set: every cached file must reproduce its halting logits and grids bit for bit.
REF = {}
for name, batch in BATCHES:
    _, pred0, q0 = run_block(batch)
    valid = correct(pred0, batch)[1]
    REF[name] = q0[valid].cpu().numpy(), pred0[valid].cpu().numpy()


def cached_pass(tag, arm, seed):
    """The pass of `arm` for `seed`, loaded from the cache or run and saved."""
    path = os.path.join(CACHE_DIR, "TRM_pass%s_seed%d.npz" % (tag, seed))
    if not os.path.exists(path):
        arrays = {}
        for name, r in run_pass(arm, seed).items():
            arrays.update({name + "__ok0": r["ok0"].astype(np.uint8), name + "__q0": r["q0"],
                           name + "__pred0": r["pred0"].astype(np.int16),
                           "%s__%s__ok" % (name, arm): r["ok"].astype(np.uint8),
                           "%s__%s__act" % (name, arm): r["act"].astype(np.uint8)})
        np.savez_compressed(path, **arrays)
    f = np.load(path)
    same = all(np.array_equal(f[name + "__q0"], REF[name][0]) and np.array_equal(f[name + "__pred0"], REF[name][1])
               for name in REF)
    assert same, "%s comes from another numerical setup (GPU, PyTorch): delete it to recompute it" % path
    return {name: dict(ok0=f[name + "__ok0"].astype(bool), q0=f[name + "__q0"], pred0=f[name + "__pred0"],
                       ok=f["%s__%s__ok" % (name, arm)].astype(bool), act=f["%s__%s__act" % (name, arm)].astype(bool))
            for name, _ in BATCHES}

In [ ]:
# Self-test, on the first set with re-opened puzzles: a restart that resamples nothing replays the standard pass bit
# for bit.
for name, batch in BATCHES:
    carry0, pred0, q0 = run_block(batch)
    active = correct(pred0, batch)[1] & (q0 < Q_THR)
    if active.any():
        break
for arm in ("restart", "restart_mirror"):
    carry = clone(carry0)
    retry(carry, torch.zeros_like(batch["inputs"], dtype=torch.bool), active, arm, torch.Generator(device=DEVICE))
    _, pred, q = run_block(batch, carry)
    assert (pred[active] == pred0[active]).all() and (q[active] == q0[active]).all()

## z instead of y, GAR and PTRM

In [ ]:
PTRM_K, PTRM_SIGMA = 100, 1.5

MIRROR = [cached_pass("MIR%03d" % ROUNDS, "restart_mirror", s) for s in SEEDS]
GAR = [cached_pass("RST%03d" % ROUNDS, "restart", s) for s in SEEDS]

PTRM = []
for s in SEEDS:
    path = os.path.join(CACHE_DIR, "ptrm", "TRM__K%d_D16_s%.2f__seed%d.npz" % (PTRM_K, PTRM_SIGMA, s))
    assert os.path.exists(path), "%s is missing: run PPBench_continue_and_PTRM first" % path
    f = np.load(path)
    assert all(np.array_equal(f[name + "__ref_q0"], REF[name][0]) for name in REF), \
        "%s comes from another numerical setup (GPU, PyTorch)" % path
    PTRM.append({name: (f[name + "__ok"].astype(bool), f[name + "__q"]) for name in REF})


def ptrm_harness(ok, q, ok0, q0, k):
    """PTRM under the harness: on a re-opened puzzle, the rollouts are read in order until one has q >= Q_THR, at
    most k of them, and the fallback returns the grid of highest q among the standard pass and the rollouts read.
    Returns the correctness per puzzle."""
    ok, q = ok[:k], q[:k]
    n = ok.shape[1]
    hit = q >= Q_THR
    last = np.where(hit.any(0), hit.argmax(0), k - 1)
    read = np.concatenate([np.ones((1, n), bool), np.arange(k)[:, None] <= last[None, :]])
    best = np.where(read, np.concatenate([q0[None, :], q]), -np.inf).argmax(0)
    return np.where(q0 < Q_THR, np.concatenate([ok0[None, :], ok])[best, np.arange(n)], ok0)


def reference(sets):
    """Correctness and halting logit of the standard pass on the puzzles of `sets`."""
    return np.concatenate([GAR[0][name]["ok0"] for name in sets]), np.concatenate([GAR[0][name]["q0"] for name in sets])


def at(passes, sets, key, r):
    """Per seed, the column r of `key` concatenated over `sets`."""
    return [np.concatenate([p[name][key][:, r] for name in sets]) for p in passes]


def paired(a, b, boot):
    """a - b in accuracy points, with a and b lists (one per seed) of per-puzzle correctness."""
    d = np.mean([x.astype(np.float64) for x in a], 0) - np.mean([x.astype(np.float64) for x in b], 0)
    lo, hi = np.percentile(d[boot].mean(1), [2.5, 97.5])
    return {"diff": 100 * d.mean(), "low": 100 * lo, "high": 100 * hi}

## Results

For each split, over the five types of its aggregate: accuracy (%) and cost (blocks per puzzle) of *z instead of y*
after `r` retries, as mean and standard deviation over the seeds, and the fraction (%) of re-opened puzzles still
searching before retry `r`. *y - z* is GAR minus *z instead of y*, and *once - noise* is *z instead of y* minus PTRM:
differences paired by puzzle and averaged over seeds, with a 95% bootstrap interval over puzzles (B = 10,000, the
seeds of a puzzle kept together). Then, at 16 and 64 retries: the failures recovered and successes lost, the puzzles
solved by only one of GAR and *z instead of y*, and the accuracy per type.

In [ ]:
R_LIST = [r for r in (1, 2, 5, 10, 16, 25, 32, 48, 50, 64) if r <= ROUNDS]

TABLES = {}
for split in ("val", "gold"):
    sets = split_sets(split)
    ok0, q0 = reference(sets)
    n_fail, n_open = (~ok0).sum(), (q0 < Q_THR).sum()
    boot = np.random.default_rng(0).integers(0, len(ok0), size=(10000, len(ok0)))
    rollouts = [[np.concatenate([p[name][i] for name in sets], 1) for i in (0, 1)] for p in PTRM]

    rows = {}
    for r in R_LIST:
        gar, mirror = at(GAR, sets, "ok", r), at(MIRROR, sets, "ok", r)
        acc = [100 * v.mean() for v in mirror]
        cost = [1.0 + sum(p[name]["act"][:, :r].sum() for name in sets) / len(ok0) for p in MIRROR]
        still_open = [sum(p[name]["act"][:, r - 1].sum() for name in sets) for p in MIRROR]
        rows[r] = {("z instead of y", "accuracy"): np.mean(acc), ("z instead of y", "sd"): np.std(acc, ddof=1),
                   ("z instead of y", "cost"): np.mean(cost),
                   ("z instead of y", "open"): 100 * np.mean(still_open) / n_open}
        rows[r].update({("y - z", k): v for k, v in paired(gar, mirror, boot).items()})
        ptrm = [ptrm_harness(ok, q, ok0, q0, r) for ok, q in rollouts]
        rows[r].update({("once - noise", k): v for k, v in paired(mirror, ptrm, boot).items()})
    table = pd.DataFrame.from_dict(rows, orient="index").rename_axis("r")

    recovered, solved_by_one = {}, {}
    for r in (16, ROUNDS):
        gar, mirror = at(GAR, sets, "ok", r), at(MIRROR, sets, "ok", r)
        rec = [(~ok0 & v).sum() for v in mirror]
        recovered[r] = {"recovered": np.mean(rec), "sd": np.std(rec, ddof=1),
                        "recovered (%)": 100 * np.mean(rec) / n_fail, "sd (%)": 100 * np.std(rec, ddof=1) / n_fail,
                        "lost": np.mean([(ok0 & ~v).sum() for v in mirror])}
        for i, s in enumerate(SEEDS):
            solved_by_one[(r, s)] = {"GAR only": (gar[i] & ~mirror[i]).sum(),
                                     "z instead of y only": (mirror[i] & ~gar[i]).sum()}

    per_type = {}
    for label, subset in [(t, [split + "_" + t]) for t in REPORT_TYPES] + [("aggregate", sets)]:
        o0 = reference(subset)[0]
        per_type[label] = {"puzzles": len(o0), "failures": (~o0).sum(), "standard pass": 100 * o0.mean(),
                           **{"%d retries" % r: np.mean([100 * v.mean() for v in at(MIRROR, subset, "ok", r)])
                              for r in (16, ROUNDS)}}

    TABLES[split] = dict(table=table, recovered=pd.DataFrame.from_dict(recovered, orient="index").rename_axis("r"),
                         solved_by_one=pd.DataFrame(solved_by_one).rename_axis(columns=["r", "seed"]),
                         per_type=pd.DataFrame.from_dict(per_type, orient="index"))

### Validation

In [ ]:
display(TABLES["val"]["table"], TABLES["val"]["recovered"], TABLES["val"]["solved_by_one"], TABLES["val"]["per_type"])

### Golden

In [ ]:
display(TABLES["gold"]["table"], TABLES["gold"]["recovered"], TABLES["gold"]["solved_by_one"],
        TABLES["gold"]["per_type"])